<a href="https://colab.research.google.com/github/workzzspace99-ai/ashwinnn/blob/main/flask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Load the Diabetes Dataset
We will download and load the Pima Indians Diabetes Database, which is commonly used to predict whether a patient has diabetes based on diagnostic measurements.

In [1]:
import pandas as pd
import numpy as np

# URL of the dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

# Define column names based on dataset documentation
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

# Load the dataset
df = pd.read_csv(url, names=columns)

# Display basic information and first few rows
print(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Preprocessing
- Identify physically impossible `0` values in features like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` and replace them with median values.
- Split the data into train and test sets.
- Scale the features using standard scaling.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Make a copy of the dataframe
data = df.copy()

# Columns where '0' is invalid and indicates missing values
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Replace 0s with NaN
data[zero_cols] = data[zero_cols].replace(0, np.nan)

# Impute NaNs with Median values of each column
for col in zero_cols:
    median_val = data[col].median()
    data[col].fillna(median_val, inplace=True)

# Split features and target
X = data.drop(columns=['Outcome'])
y = data['Outcome']

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Shape of train set:", X_train_scaled.shape)
print("Shape of test set:", X_test_scaled.shape)
print("Missing values remaining:\n", data.isnull().sum())

Shape of train set: (614, 8)
Shape of test set: (154, 8)
Missing values remaining:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


/tmp/ipykernel_795/3409564883.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(median_val, inplace=True)


## 3. Train and Compare Machine Learning Models
We will fit multiple machine learning models to the preprocessed training set, make predictions on the test set, and compare their performance using metrics such as Accuracy, Precision, Recall, and F1-Score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Evaluate and store results
results = []
for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    # Predict
    preds = model.predict(X_test_scaled)

    # Calculate metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

# Convert results to DataFrame for comparison
comparison_df = pd.DataFrame(results)
display(comparison_df.sort_values(by='Accuracy', ascending=False))

In [ ]:
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

# Train the final model on all scaled data
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X)
final_model = LogisticRegression(random_state=42)
final_model.fit(X_scaled, y)

# Save the model and scaler to disk
joblib.dump(final_model, 'diabetes_model.pkl')
joblib.dump(final_scaler, 'scaler.pkl')

# Create templates directory
os.makedirs('templates', exist_ok=True)

# Create the HTML form
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Diabetes Prediction System</title>
    <style>
        body { font-family: Arial, sans-serif; background-color: #f4f7f6; margin: 0; padding: 20px; }
        .container { max-width: 500px; background: white; padding: 30px; margin: auto; border-radius: 10px; box-shadow: 0px 0px 10px rgba(0,0,0,0.1); }
        h2 { text-align: center; color: #333; }
        .form-group { margin-bottom: 15px; }
        label { display: block; margin-bottom: 5px; color: #666; }
        input { width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px; box-sizing: border-box; }
        input[type="submit"] { background-color: #28a745; color: white; border: none; font-size: 16px; cursor: pointer; margin-top: 10px; }
        input[type="submit"]:hover { background-color: #218838; }
        .result { text-align: center; margin-top: 20px; font-size: 18px; font-weight: bold; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Diabetes Predictor</h2>
        <form action="/predict" method="POST">
            <div class="form-group"><label>Pregnancies</label><input type="number" name="Pregnancies" value="1" required></div>
            <div class="form-group"><label>Glucose</label><input type="number" name="Glucose" value="120" required></div>
            <div class="form-group"><label>Blood Pressure</label><input type="number" name="BloodPressure" value="70" required></div>
            <div class="form-group"><label>Skin Thickness (mm)</label><input type="number" name="SkinThickness" value="20" required></div>
            <div class="form-group"><label>Insulin</label><input type="number" name="Insulin" value="80" required></div>
            <div class="form-group"><label>BMI</label><input type="number" step="0.1" name="BMI" value="25.4" required></div>
            <div class="form-group"><label>Diabetes Pedigree Function</label><input type="number" step="0.001" name="DiabetesPedigreeFunction" value="0.35" required></div>
            <div class="form-group"><label>Age</label><input type="number" name="Age" value="30" required></div>
            <input type="submit" value="Predict">
        </form>
        {% if prediction_text %}
            <div class="result" style="color: {{ 'red' if 'Positive' in prediction_text else 'green' }};">
                {{ prediction_text }}
            </div>
        {% endif }
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

# Create the Flask application script
flask_app = """
from flask import Flask, request, render_template
import joblib
import numpy as np

app = Flask(__name__)

# Load assets
model = joblib.load('diabetes_model.pkl')
scaler = joblib.load('scaler.pkl')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    # Retrieve values from form
    features = [
        float(request.form['Pregnancies']),
        float(request.form['Glucose']),
        float(request.form['BloodPressure']),
        float(request.form['SkinThickness']),
        float(request.form['Insulin']),
        float(request.form['BMI']),
        float(request.form['DiabetesPedigreeFunction']),
        float(request.form['Age'])
    ]

    # Preprocess and scale inputs
    features_arr = np.array([features])
    features_scaled = scaler.transform(features_arr)

    # Predict
    prediction = model.predict(features_scaled)[0]
    probability = model.predict_proba(features_scaled)[0][1]

    if prediction == 1:
        result = f"Prediction: Positive for Diabetes (Probability: {probability*100:.1f}%)"
    else:
        result = f"Prediction: Negative for Diabetes (Probability: {probability*100:.1f}%)"

    return render_template('index.html', prediction_text=result)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

with open('app.py', 'w') as f:
    f.write(flask_app)

print("Saved web app assets and configuration successfully!")